# EXPERIMENT TAB - Data Preprocessing Pipeline (9 Steps Only)

**Fixed to run without errors. Uses pandas, numpy, sklearn only.**

**Raw data path**: `../../corndata_raw/corn_data.csv` (adjust if needed)

**Stops at Step 9**: Save `preprocessed_corn_data.csv`

## 1. Import Library (pandas, numpy, sklearn only)

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

## 2. Load Raw Data

In [16]:
# Load from corndata_raw file (relative path from workflow/)
df = pd.read_csv('../../corndata_raw')
print(f'Data loaded: {df.shape}')
print(df.head())

Data loaded: (422, 22)
         County   Farmer    Education Gender Age bracket  Household size  \
0  TAITA TAVETA   fmr_65  Certificate   Male       36-45               7   
1  TAITA TAVETA   fmr_77  Certificate   Male       36-45               7   
2  TAITA TAVETA   fmr_89  Certificate   Male       36-45               7   
3  TAITA TAVETA  fmr_102  Certificate   Male       36-45               7   
4  TAITA TAVETA   fmr_25  Certificate   Male       46-55               3   

   Crop  Acreage  Fertilizer amount  Laborers  ...  Water source  \
0  corn     2.00                 50         2  ...          Rain   
1  corn     0.25                 50         2  ...          Rain   
2  corn     3.00                251         2  ...          Rain   
3  corn     1.50                300         3  ...          Rain   
4  corn      NaN                 50         2  ...          Rain   

  Main credit source Crop insurance Farm records Main advisory source  \
0      Credit groups             No   

## 3. EDA (df.info(), df.describe())

In [17]:
print('INFO:')
df.info()
print('\nDESCRIBE:')
print(df.describe())

INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 422 entries, 0 to 421
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   County                422 non-null    object 
 1   Farmer                422 non-null    object 
 2   Education             396 non-null    object 
 3   Gender                422 non-null    object 
 4   Age bracket           422 non-null    object 
 5   Household size        422 non-null    int64  
 6   Crop                  422 non-null    object 
 7   Acreage               351 non-null    float64
 8   Fertilizer amount     422 non-null    int64  
 9   Laborers              422 non-null    int64  
 10  Yield                 422 non-null    int64  
 11  Power source          422 non-null    object 
 12  Water source          422 non-null    object 
 13  Main credit source    422 non-null    object 
 14  Crop insurance        422 non-null    object 
 15  Farm records     

## 4. Missing Value Handling

In [18]:
# Check missing
print('Missing values:')
print(df.isnull().sum())

# Handle: Acreage (median), Education (mode)
if 'Acreage' in df.columns:
    median_acreage = df['Acreage'].median()
    df['Acreage'].fillna(median_acreage, inplace=True)
if 'Education' in df.columns:
    mode_education = df['Education'].mode()[0]
    df['Education'].fillna(mode_education, inplace=True)
print('\nAfter handling:')
print(df.isnull().sum())

Missing values:
County                   0
Farmer                   0
Education               26
Gender                   0
Age bracket              0
Household size           0
Crop                     0
Acreage                 71
Fertilizer amount        0
Laborers                 0
Yield                    0
Power source             0
Water source             0
Main credit source       0
Crop insurance           0
Farm records             0
Main advisory source     0
Extension provider       0
Advisory format          0
Advisory language        0
Latitude                 0
Longitude                0
dtype: int64

After handling:
County                  0
Farmer                  0
Education               0
Gender                  0
Age bracket             0
Household size          0
Crop                    0
Acreage                 0
Fertilizer amount       0
Laborers                0
Yield                   0
Power source            0
Water source            0
Main credit source    

## 5. Remove Duplicates

In [19]:
print(f'Duplicates before: {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'Duplicates after: {df.duplicated().sum()}')

Duplicates before: 0
Duplicates after: 0


## 6. One-Hot Encoding

In [20]:
categorical_cols = df.select_dtypes(include='object').columns
exclude_cols = ['County', 'Farmer', 'Crop', 'Power source', 'Water source', 'Crop insurance']
cols_to_encode = [col for col in categorical_cols if col not in exclude_cols]
df_encoded = pd.get_dummies(df, columns=cols_to_encode, drop_first=True, dtype=int)
print(f'Encoded shape: {df_encoded.shape}')
print(df_encoded.head())

Encoded shape: (422, 35)
         County   Farmer  Household size  Crop  Acreage  Fertilizer amount  \
0  TAITA TAVETA   fmr_65               7  corn     2.00                 50   
1  TAITA TAVETA   fmr_77               7  corn     0.25                 50   
2  TAITA TAVETA   fmr_89               7  corn     3.00                251   
3  TAITA TAVETA  fmr_102               7  corn     1.50                300   
4  TAITA TAVETA   fmr_25               3  corn     0.50                 50   

   Laborers  Yield Power source Water source  ...  \
0         2    300       Manual         Rain  ...   
1         2    270       Manual         Rain  ...   
2         2    270       Manual         Rain  ...   
3         3    200       Manual         Rain  ...   
4         2    180       Manual         Rain  ...   

  Main advisory source_Internet  Main advisory source_Public gatherings  \
0                             0                                       0   
1                             0      

## 7. Min-Max Normalization

In [21]:
scaler = MinMaxScaler()
numerical_cols = ['Household size', 'Acreage', 'Fertilizer amount', 'Laborers', 'Yield', 'Latitude', 'Longitude']
available_numerical = [col for col in numerical_cols if col in df_encoded.columns]
df_encoded[available_numerical] = scaler.fit_transform(df_encoded[available_numerical])
print('Normalized columns:', available_numerical)
print(df_encoded[available_numerical].head())

Normalized columns: ['Household size', 'Acreage', 'Fertilizer amount', 'Laborers', 'Yield', 'Latitude', 'Longitude']
   Household size   Acreage  Fertilizer amount  Laborers     Yield  Latitude  \
0            0.75  0.466667           0.062500  0.000000  0.454545  0.461538   
1            0.75  0.000000           0.062500  0.000000  0.400000  0.846154   
2            0.75  0.733333           0.481250  0.000000  0.400000  0.589744   
3            0.75  0.333333           0.583333  0.166667  0.272727  0.641026   
4            0.25  0.066667           0.062500  0.000000  0.236364  0.641026   

   Longitude  
0   0.222222  
1   0.361111  
2   0.277778  
3   0.277778  
4   0.166667  


## 8. Outlier Handling (IQR Capping)

In [22]:
for col in available_numerical:
    Q1 = df_encoded[col].quantile(0.25)
    Q3 = df_encoded[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_encoded[col] = df_encoded[col].clip(lower=lower, upper=upper)
    print(f'{col}: [{lower:.4f}, {upper:.4f}]')
print('\nOutliers handled.')

Household size: [-0.1250, 0.8750]
Acreage: [-0.1333, 0.4000]
Fertilizer amount: [-0.1458, 0.2708]
Laborers: [-0.2500, 0.4167]
Yield: [-0.5364, 1.1364]
Latitude: [-0.1154, 1.2179]
Longitude: [-0.0556, 0.6111]

Outliers handled.


## 9. Save to preprocessed_corn_data.csv ← END

In [24]:
df_encoded.to_csv('preprocessed_corn_data.csv', index=False)
print('✅ Saved preprocessed_corn_data.csv')
print(f'Final shape: {df_encoded.shape}')

✅ Saved preprocessed_corn_data.csv
Final shape: (422, 35)
